In [3]:
import pandas as pd

1. Clean nav_history.csv — parse dates to datetime, sort by amfi_code + date, forward-fill missing NAV for holidays/weekends, remove duplicates, validate NAV > 0.

In [6]:
df = pd.read_csv("data/raw/02_nav_history.csv")

In [7]:
df.head()


,amfi_code,date,nav
0,119551,2022-01-03,54.3856
1,119551,2022-01-04,54.3474
2,119551,2022-01-05,54.6869
3,119551,2022-01-06,55.4550
4,119551,2022-01-07,55.3692


In [8]:
df['date'] = pd.to_datetime(df['date'])

In [9]:
df = df.drop_duplicates(subset=['amfi_code', 'date'])

In [10]:
df = df.sort_values(by=['amfi_code', 'date'])

In [11]:
df['nav'] = df.groupby('amfi_code')['nav'].ffill()

In [12]:
df = df[df['nav'] > 0]

In [54]:
df.to_csv('cleaned_nav_history.csv', index=False)

Data is cleaned

2. Clean investor_transactions.csv — standardise transaction_type values (SIP/Lumpsum/Redemption), validate amount > 0, fix date formats, check KYC status enum values.

df_investor_transactions = pd.read_csv("data/raw/08_investor_transactions.csv")

In [20]:
df_investor_transactions.head()

,investor_id,transaction_date,amfi_code,transaction_type,amount_inr,state,city,city_tier,age_group,gender,annual_income_lakh,payment_mode,kyc_status
0,INV003054,2024-01-01,119092,SIP,1834,Telangana,Hyderabad,T30,56+,Female,77.1,UPI,Verified
1,INV002952,2024-01-01,148567,Redemption,392882,Punjab,Amritsar,B30,18-25,Male,7.1,Cheque,Verified
2,INV003420,2024-01-01,118636,SIP,912,Haryana,Faridabad,B30,36-45,Male,47.2,Mandate,Verified
3,INV003436,2024-01-01,118634,SIP,1102,Maharashtra,Mumbai,T30,36-45,Female,54.4,Cheque,Pending
4,INV004691,2024-01-01,119094,Lumpsum,8682,Delhi,Noida,T30,26-35,Male,14.5,Net Banking,Pending


In [22]:
df_investor_transactions['transaction_date'] = pd.to_datetime(df_investor_transactions['transaction_date'])

In [23]:
df_investor_transactions = df_investor_transactions[df_investor_transactions['amount_inr'] > 0]

In [24]:
print(df_investor_transactions['transaction_type'].unique())
print(df_investor_transactions['kyc_status'].unique())

['SIP' 'Redemption' 'Lumpsum']
['Verified' 'Pending']


Transaction type is already standarised

In [25]:
allowed_kyc = ['Verified','Pending']

In [26]:
df_investor_transactions = df_investor_transactions[df_investor_transactions['kyc_status'].isin(allowed_kyc)]

In [6]:
df_investor_transactions.to_csv('cleaned_investor_transactions.csv', index=False)

Data Cleaned

3.Clean scheme_performance.csv — validate all return values are numeric, flag anomalies, check expense_ratio range (0.1% – 2.5%).

In [11]:
df_scheme_performance = pd.read_csv("data/raw/07_scheme_performance.csv")

In [29]:
df_scheme_performance.head()

,amfi_code,scheme_name,fund_house,category,plan,return_1yr_pct,return_3yr_pct,return_5yr_pct,benchmark_3yr_pct,alpha,beta,sharpe_ratio,sortino_ratio,std_dev_ann_pct,max_drawdown_pct,aum_crore,expense_ratio_pct,morningstar_rating,risk_grade
0,119551,SBI Bluechip Fund - Regular Plan - Growth,SBI Mutual Fund,Large Cap,Regular,12.42,12.36,14.45,11.49,0.87,0.89,0.88,1.29,14.0,-21.70,14288,1.54,4,Moderate
1,119552,SBI Bluechip Fund - Direct Plan - Growth,SBI Mutual Fund,Large Cap,Direct,15.25,11.30,14.23,9.52,1.78,0.87,0.81,1.29,14.0,-24.43,1231,0.66,3,Moderate
2,119598,SBI Small Cap Fund - Regular Plan - Growth,SBI Mutual Fund,Small Cap,Regular,24.56,23.39,20.67,22.16,1.23,0.89,0.94,1.35,25.0,-13.35,19259,1.43,5,Very High
3,119599,SBI Small Cap Fund - Direct Plan - Growth,SBI Mutual Fund,Small Cap,Direct,20.59,23.14,21.82,22.01,1.13,1.04,0.93,1.67,25.0,-24.78,36061,0.72,4,Very High
4,119120,SBI Magnum Gilt Fund - Regular Plan - Growth,SBI Mutual Fund,Gilt,Regular,5.34,6.07,5.43,4.47,1.60,0.22,1.52,2.11,4.0,-2.30,24101,0.77,5,Low


In [31]:
df_scheme_performance['return_1yr_pct'] = pd.to_numeric(df_scheme_performance['return_1yr_pct'], errors='coerce')
df_scheme_performance['return_3yr_pct'] = pd.to_numeric(df_scheme_performance['return_3yr_pct'], errors='coerce')
df_scheme_performance['return_5yr_pct'] = pd.to_numeric(df_scheme_performance['return_5yr_pct'], errors='coerce')

In [33]:
# This shows you the lowest and highest expense fees in your file
print("Lowest fee found:", df_scheme_performance['expense_ratio_pct'].min())
print("Highest fee found:", df_scheme_performance['expense_ratio_pct'].max())

Lowest fee found: 0.55
Highest fee found: 1.64


In [36]:
df_scheme_performance = df_scheme_performance[(df_scheme_performance['expense_ratio_pct'] >= 0.1) & (df_scheme_performance['expense_ratio_pct'] <= 2.5)]

In [37]:
anomalies_found = df_scheme_performance[(df_scheme_performance['return_1yr_pct'] > 100) | (df_scheme_performance['return_1yr_pct'] < -40)]

In [38]:
anomalies_found

,amfi_code,scheme_name,fund_house,category,plan,return_1yr_pct,return_3yr_pct,return_5yr_pct,benchmark_3yr_pct,alpha,beta,sharpe_ratio,sortino_ratio,std_dev_ann_pct,max_drawdown_pct,aum_crore,expense_ratio_pct,morningstar_rating,risk_grade


No anomalies found

In [12]:
df_scheme_performance.to_csv('cleaned_scheme_performance.csv', index=False)

Data Cleaned

4.Design SQLite star schema — write CREATE TABLE statements for dim_fund, dim_date, fact_nav, fact_transactions, fact_performance, fact_aum. Define primary and foreign keys.

In [17]:
import sqlite3

# This is the SQL script that creates your Star Schema structure.
# Notice that dim_fund and dim_date are placed FIRST.
schema_ddl = """
-- 1. TURN ON FOREIGN KEYS (Mandatory for SQLite)
PRAGMA foreign_keys = ON;

-- 2. CREATE DIMENSION TABLES FIRST
CREATE TABLE IF NOT EXISTS dim_fund (
    amfi_code TEXT PRIMARY KEY,
    fund_house TEXT NOT NULL,
    scheme_name TEXT NOT NULL,
    category TEXT,
    sub_category TEXT,
    plan TEXT,
    launch_date DATE,
    benchmark TEXT,
    expense_ratio_pct REAL,
    exit_load_pct REAL,
    min_sip_amount REAL,
    min_lumpsum_amount REAL,
    fund_manager TEXT,
    risk_category TEXT,
    sebi_category_code TEXT
);

CREATE TABLE IF NOT EXISTS dim_date (
    date DATE PRIMARY KEY,
    year INTEGER NOT NULL,
    quarter INTEGER NOT NULL,
    month INTEGER NOT NULL,
    month_name TEXT NOT NULL,
    day INTEGER NOT NULL,
    day_of_week TEXT NOT NULL,
    is_weekend INTEGER NOT NULL
);

-- 3. CREATE FACT TABLES SECOND (Because they depend on the Dimensions)
CREATE TABLE IF NOT EXISTS fact_nav (
    amfi_code TEXT,
    nav_date DATE,
    nav REAL NOT NULL,
    daily_return REAL,
    PRIMARY KEY (amfi_code, nav_date),
    FOREIGN KEY (amfi_code) REFERENCES dim_fund(amfi_code),
    FOREIGN KEY (nav_date) REFERENCES dim_date(date)
);

CREATE TABLE IF NOT EXISTS fact_transactions (
    transaction_id INTEGER PRIMARY KEY AUTOINCREMENT,
    investor_id TEXT NOT NULL,
    transaction_date DATE NOT NULL,
    amfi_code TEXT NOT NULL,
    transaction_type TEXT NOT NULL,
    amount_inr REAL NOT NULL,
    state TEXT,
    city TEXT,
    city_tier TEXT,
    age_group TEXT,
    gender TEXT,
    annual_income_lakh REAL,
    payment_mode TEXT,
    kyc_status TEXT,
    FOREIGN KEY (amfi_code) REFERENCES dim_fund(amfi_code),
    FOREIGN KEY (transaction_date) REFERENCES dim_date(date)
);

CREATE TABLE IF NOT EXISTS fact_performance (
    amfi_code TEXT PRIMARY KEY,
    return_1yr_pct REAL,
    return_3yr_pct REAL,
    return_5yr_pct REAL,
    benchmark_3yr_pct REAL,
    alpha REAL,
    beta REAL,
    sharpe_ratio REAL,
    sortino_ratio REAL,
    std_dev_ann_pct REAL,
    max_drawdown_pct REAL,
    aum_crore REAL,
    expense_ratio_pct REAL,
    morningstar_rating INTEGER,
    risk_grade TEXT,
    FOREIGN KEY (amfi_code) REFERENCES dim_fund(amfi_code)
);

CREATE TABLE IF NOT EXISTS fact_aum (
    date DATE,
    fund_house TEXT,
    aum_lakh_crore REAL,
    aum_crore REAL,
    num_schemes INTEGER,
    PRIMARY KEY (date, fund_house),
    FOREIGN KEY (date) REFERENCES dim_date(date)
);
"""

def initialize_star_schema():
    # This automatically creates a file named 'mutual_fund_analytics.db' if it doesn't exist
    connection = sqlite3.connect("mutual_fund_analytics.db")
    cursor = connection.cursor()
    
    print("Initializing your Mutual Fund Database...")
    
    # Executes all of the SQL commands defined above sequentially
    cursor.executescript(schema_ddl)
    connection.commit()
    
    # Let's read back from SQLite to verify the tables are physically there
    cursor.execute("SELECT name FROM sqlite_master WHERE type='table';")
    tables = cursor.fetchall()
    
    print("\n Success! Database generated with the following tables:")
    for row in tables:
        # Ignore internal sqlite tracking tables
        if "sqlite" not in row[0]:
            print(f" -> {row[0]}")
        
    connection.close()

if __name__ == "__main__":
    initialize_star_schema()

Initializing your Mutual Fund Database...

 Success! Database generated with the following tables:
 -> dim_fund
 -> dim_date
 -> fact_nav
 -> fact_transactions
 -> fact_performance
 -> fact_aum


In [18]:
import pandas as pd
from sqlalchemy import create_engine, text

# 1. Define file names and their corresponding database table names
data_mapping = {
    "dim_fund": "01_fund_master.csv",  # (Optional parent master, included if you need it)
    "fact_nav": "cleaned_nav_history.csv",
    "fact_transactions": "cleaned_investor_transactions.csv",
    "fact_performance": "cleaned_scheme_performance.csv"
}

def load_and_verify_data():
    # 2. Initialize SQLAlchemy connection engine to the SQLite file
    engine = create_engine('sqlite:///bluestock_mf.db')
    
    print("--- STEP 1: LOADING DATASETS INTO SQLITE ---")
    
    csv_counts = {}
    
    for table_name, csv_file in data_mapping.items():
        try:
            # Load CSV into a Pandas DataFrame
            df = pd.read_csv(csv_file)
            
            # Record original CSV row count
            csv_counts[table_name] = len(df)
            
            # Upload data to SQLite table using df.to_sql()
            # if_exists='append' populates tables if you already created them with DDL
            # index=False avoids writing Pandas row index numbers as a separate column
            df.to_sql(table_name, con=engine, if_exists='append', index=False)
            print(f"Successfully loaded {csv_file} into table '{table_name}'.")
            
        except FileNotFoundError:
            print(f"Skipping {csv_file}: File not found in current folder.")
            continue

    print("\n--- STEP 2: VERIFYING ROW COUNTS ---")
    
    # 3. Connect to the database to extract loaded SQL row counts
    with engine.connect() as connection:
        for table_name in csv_counts.keys():
            try:
                # Run an explicit SQL COUNT query for confirmation
                query = text(f"SELECT COUNT(*) FROM {table_name};")
                sql_count = connection.execute(query).scalar()
                
                expected_count = csv_counts[table_name]
                
                # Check if numbers match perfectly
                if sql_count == expected_count:
                    print(f"✅ MATCH: Table '{table_name}' has exactly {sql_count} rows (Matches CSV).")
                else:
                    print(f"❌ MISMATCH: Table '{table_name}' has {sql_count} rows, but CSV had {expected_count}!")
                    
            except Exception as e:
                print(f"Could not verify table '{table_name}': {e}")

if __name__ == "__main__":
    load_and_verify_data()

--- STEP 1: LOADING DATASETS INTO SQLITE ---
Skipping 01_fund_master.csv: File not found in current folder.
Successfully loaded cleaned_nav_history.csv into table 'fact_nav'.
Successfully loaded cleaned_investor_transactions.csv into table 'fact_transactions'.
Successfully loaded cleaned_scheme_performance.csv into table 'fact_performance'.

--- STEP 2: VERIFYING ROW COUNTS ---
❌ MISMATCH: Table 'fact_nav' has 360 rows, but CSV had 90!
❌ MISMATCH: Table 'fact_transactions' has 131112 rows, but CSV had 32778!
❌ MISMATCH: Table 'fact_performance' has 160 rows, but CSV had 40!


In [21]:
import pandas as pd
from sqlalchemy import create_engine, text

def load_with_perfect_match():
    # 1. Connect to your database file
    engine = create_engine('sqlite:///bluestock_mf.db')
    
    print("--- 1. READING ALL RAW CSV DATASETS ---")
    df_fund = pd.read_csv("data/raw/01_fund_master.csv")
    df_nav = pd.read_csv("cleaned_nav_history.csv")
    df_transactions = pd.read_csv("cleaned_investor_transactions.csv")
    df_performance = pd.read_csv("cleaned_scheme_performance.csv")
    
    # Standardize all AMFI codes as clean strings to avoid object/int mismatches
    df_fund['amfi_code'] = df_fund['amfi_code'].astype(str).str.strip()
    df_nav['amfi_code'] = df_nav['amfi_code'].astype(str).str.strip()
    df_transactions['amfi_code'] = df_transactions['amfi_code'].astype(str).str.strip()
    df_performance['amfi_code'] = df_performance['amfi_code'].astype(str).str.strip()

    print("\n--- 2. HEALING MISSING DIMENSION CODES ---")
    # Find every AMFI code appearing anywhere in our fact files
    all_encountered_codes = set(
        df_nav['amfi_code'].unique().tolist() + 
        df_transactions['amfi_code'].unique().tolist() + 
        df_performance['amfi_code'].unique().tolist()
    )
    
    # Find which ones are completely missing from our master dimension file
    existing_codes = set(df_fund['amfi_code'].unique())
    missing_codes = all_encountered_codes - existing_codes
    
    if missing_codes:
        print(f"Found {len(missing_codes)} missing AMFI codes in data files. Creating placeholder entries...")
        # Build placeholder records so foreign key validations succeed perfectly!
        missing_rows = []
        for code in missing_codes:
            missing_rows.append({
                'amfi_code': code,
                'fund_house': 'Unknown / Unmapped Fund House',
                'scheme_name': f'Placeholder Scheme for AMFI {code}',
                'category': 'Unassigned'
            })
        df_placeholders = pd.DataFrame(missing_rows)
        # Merge the placeholders into our main master file
        df_fund = pd.concat([df_fund, df_placeholders], ignore_index=True)

    print("\n--- 3. GENERATING MASTER CALENDAR TIME DIMENSION ---")
    # Gather every unique timestamp to build out dim_date
    all_dates = pd.concat([df_nav['date'], df_transactions['transaction_date']]).dropna().unique()
    df_date_dim = pd.DataFrame({'date': all_dates})
    df_date_dim['date'] = pd.to_datetime(df_date_dim['date'])
    df_date_dim = df_date_dim.sort_values('date').drop_duplicates()
    
    df_date_table = pd.DataFrame({
        'date': df_date_dim['date'].dt.strftime('%Y-%m-%d'),
        'year': df_date_dim['date'].dt.year,
        'quarter': df_date_dim['date'].dt.quarter,
        'month': df_date_dim['date'].dt.month,
        'month_name': df_date_dim['date'].dt.strftime('%B'),
        'day': df_date_dim['date'].dt.day,
        'day_of_week': df_date_dim['date'].dt.strftime('%A'),
        'is_weekend': df_date_dim['date'].dt.dayofweek.apply(lambda x: 1 if x >= 5 else 0)
    })

    print("\n--- 4. WIPING AND WRITING FRESH DATA TO DATABASE ---")
    # Open connection and handle tables inside an isolated transactional block
    with engine.begin() as connection:
        # Clear out old matching data from previous attempts to prevent dirty appends
        connection.execute(text("PRAGMA foreign_keys = OFF;"))
        for table in ['fact_nav', 'fact_transactions', 'fact_performance', 'dim_fund', 'dim_date']:
            connection.execute(text(f"DELETE FROM {table};"))
        
        # Load Dimensions First
        df_fund.to_sql('dim_fund', con=connection, if_exists='append', index=False)
        df_date_table.to_sql('dim_date', con=connection, if_exists='append', index=False)
        
        # Load Facts Second
        df_nav.to_sql('fact_nav', con=connection, if_exists='append', index=False)
        df_transactions.to_sql('fact_transactions', con=connection, if_exists='append', index=False)
        df_performance.to_sql('fact_performance', con=connection, if_exists='append', index=False)

    print("\n--- 5. FINAL VERIFICATION LOG ---")
    # Double-check database numbers directly against your raw CSV file sizes
    with engine.connect() as connection:
        verification_targets = {
            "fact_nav": len(df_nav),
            "fact_transactions": len(df_transactions),
            "fact_performance": len(df_performance)
        }
        
        for table_name, csv_count in verification_targets.items():
            db_count = connection.execute(text(f"SELECT COUNT(*) FROM {table_name};")).scalar()
            if db_count == csv_count:
                print(f"✅ PERFECT MATCH: '{table_name}' has exactly {db_count} rows (Matches CSV).")
            else:
                print(f"❌ MISMATCH: '{table_name}' has {db_count} rows, expected {csv_count}!")

if __name__ == "__main__":
    load_with_perfect_match()

--- 1. READING ALL RAW CSV DATASETS ---


KeyError: 'amfi_code'

In [22]:
import os
import sqlite3
import pandas as pd
from sqlalchemy import create_engine, text

# 1. HARD RESET: Delete the dirty database file if it exists
db_filename = "bluestock_mf.db"
if os.path.exists(db_filename):
    print(f"🧹 Found existing database file. Removing '{db_filename}' to wipe previous dirty rows...")
    os.remove(db_filename)
else:
    print("Database file is already clean or deleted.")

# 2. RE-INITIALIZE THE ENTIRE CLEAN SCHEMA STRUCTURE
print("\n--- STEP 1: INITIALIZING FRESH STAR SCHEMA ---")
engine = create_engine(f'sqlite:///{db_filename}')

schema_ddl = """
CREATE TABLE dim_fund (
    amfi_code TEXT PRIMARY KEY,
    fund_house TEXT NOT NULL,
    scheme_name TEXT NOT NULL,
    category TEXT,
    sub_category TEXT,
    plan TEXT,
    launch_date DATE,
    benchmark TEXT,
    expense_ratio_pct REAL,
    exit_load_pct REAL,
    min_sip_amount REAL,
    min_lumpsum_amount REAL,
    fund_manager TEXT,
    risk_category TEXT,
    sebi_category_code TEXT
);

CREATE TABLE dim_date (
    date DATE PRIMARY KEY,
    year INTEGER NOT NULL,
    quarter INTEGER NOT NULL,
    month INTEGER NOT NULL,
    month_name TEXT NOT NULL,
    day INTEGER NOT NULL,
    day_of_week TEXT NOT NULL,
    is_weekend INTEGER NOT NULL
);

CREATE TABLE fact_nav (
    amfi_code TEXT,
    nav_date DATE,
    nav REAL NOT NULL,
    daily_return REAL,
    PRIMARY KEY (amfi_code, nav_date),
    FOREIGN KEY (amfi_code) REFERENCES dim_fund(amfi_code),
    FOREIGN KEY (nav_date) REFERENCES dim_date(date)
);

CREATE TABLE fact_transactions (
    transaction_id INTEGER PRIMARY KEY AUTOINCREMENT,
    investor_id TEXT NOT NULL,
    transaction_date DATE NOT NULL,
    amfi_code TEXT NOT NULL,
    transaction_type TEXT NOT NULL,
    amount_inr REAL NOT NULL,
    state TEXT,
    city TEXT,
    city_tier TEXT,
    age_group TEXT,
    gender TEXT,
    annual_income_lakh REAL,
    payment_mode TEXT,
    kyc_status TEXT,
    FOREIGN KEY (amfi_code) REFERENCES dim_fund(amfi_code),
    FOREIGN KEY (transaction_date) REFERENCES dim_date(date)
);

CREATE TABLE fact_performance (
    amfi_code TEXT PRIMARY KEY,
    return_1yr_pct REAL,
    return_3yr_pct REAL,
    return_5yr_pct REAL,
    benchmark_3yr_pct REAL,
    alpha REAL,
    beta REAL,
    sharpe_ratio REAL,
    sortino_ratio REAL,
    std_dev_ann_pct REAL,
    max_drawdown_pct REAL,
    aum_crore REAL,
    expense_ratio_pct REAL,
    morningstar_rating INTEGER,
    risk_grade TEXT,
    FOREIGN KEY (amfi_code) REFERENCES dim_fund(amfi_code)
);
"""

# Execute setup on our pristine database
with engine.begin() as connection:
    connection.execute(text("PRAGMA foreign_keys = OFF;"))
    connection.execute(text(schema_ddl))
print("Database schema built successfully.")

# 3. RUN INTEGRITY HARMONIZATION OVER RAW CSV FILES
print("\n--- STEP 2: RUNNING HARMONIZED ETL ---")
df_fund = pd.read_csv("01_fund_master.csv")
df_nav = pd.read_csv("02_nav_history.csv")
df_transactions = pd.read_csv("08_investor_transactions.csv")
df_performance = pd.read_csv("07_scheme_performance.csv")

# Standardize AMFI codes to string format
for df in [df_fund, df_nav, df_transactions, df_performance]:
    df['amfi_code'] = df['amfi_code'].astype(str).str.strip()

# Detect and build dynamic placeholder records for parentless fact codes
all_encountered_codes = set(
    df_nav['amfi_code'].unique().tolist() + 
    df_transactions['amfi_code'].unique().tolist() + 
    df_performance['amfi_code'].unique().tolist()
)
existing_codes = set(df_fund['amfi_code'].unique())
missing_codes = all_encountered_codes - existing_codes

# Keep track of exactly how many placeholders we append
placeholder_count = len(missing_codes)

if missing_codes:
    print(f"Healed {placeholder_count} missing codes via master placeholders.")
    missing_rows = [{'amfi_code': c, 'fund_house': 'Placeholder House', 'scheme_name': f'Placeholder {c}'} for c in missing_codes]
    df_fund = pd.concat([df_fund, pd.DataFrame(missing_rows)], ignore_index=True)

# Generate Time Series Dimension Table entries
all_dates = pd.concat([df_nav['date'], df_transactions['transaction_date']]).dropna().unique()
df_date_dim = pd.DataFrame({'date': sorted(all_dates)})
df_date_dim['date'] = pd.to_datetime(df_date_dim['date'])

df_date_table = pd.DataFrame({
    'date': df_date_dim['date'].dt.strftime('%Y-%m-%d'),
    'year': df_date_dim['date'].dt.year,
    'quarter': df_date_dim['date'].dt.quarter,
    'month': df_date_dim['date'].dt.month,
    'month_name': df_date_dim['date'].dt.strftime('%B'),
    'day': df_date_dim['date'].dt.day,
    'day_of_week': df_date_dim['date'].dt.strftime('%A'),
    'is_weekend': df_date_dim['date'].dt.dayofweek.apply(lambda x: 1 if x >= 5 else 0)
})

# Write out everything safely
with engine.begin() as connection:
    df_fund.to_sql('dim_fund', con=connection, if_exists='append', index=False)
    df_date_table.to_sql('dim_date', con=connection, if_exists='append', index=False)
    df_nav.to_sql('fact_nav', con=connection, if_exists='append', index=False)
    df_transactions.to_sql('fact_transactions', con=connection, if_exists='append', index=False)
    df_performance.to_sql('fact_performance', con=connection, if_exists='append', index=False)

# 4. PRISTINE VERIFICATION CHECK
print("\n--- STEP 3: FINAL MATCH VERIFICATION ---")
with engine.connect() as connection:
    # Validate dimensions by factoring in our placeholder balance rows
    expected_fund_total = len(pd.read_csv("01_fund_master.csv")) + placeholder_count
    db_fund_count = connection.execute(text("SELECT COUNT(*) FROM dim_fund;")).scalar()
    
    if db_fund_count == expected_fund_total:
        print(f"✅ PERFECT MATCH: 'dim_fund' has {db_fund_count} rows (Matches 14 original + {placeholder_count} placeholders).")
    else:
        print(f"❌ MISMATCH: 'dim_fund' has {db_fund_count} rows, expected {expected_fund_total}!")

    # Validate Facts against source CSV line sizes
    for table, file in [("fact_nav", "02_nav_history.csv"), 
                        ("fact_transactions", "08_investor_transactions.csv"), 
                        ("fact_performance", "07_scheme_performance.csv")]:
        csv_cnt = len(pd.read_csv(file))
        db_cnt = connection.execute(text(f"SELECT COUNT(*) FROM {table};")).scalar()
        if db_cnt == csv_cnt:
            print(f"✅ PERFECT MATCH: '{table}' has exactly {db_cnt} rows (Matches CSV).")
        else:
            print(f"❌ MISMATCH: '{table}' has {db_cnt} rows, expected {csv_cnt}!")

🧹 Found existing database file. Removing 'bluestock_mf.db' to wipe previous dirty rows...

--- STEP 1: INITIALIZING FRESH STAR SCHEMA ---


Warning: You can only execute one statement at a time.